# Notebook to check the ETL data

In [1]:
import pandas as pd
import os

In [2]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [81]:
## We load datasets

data_folder_path = os.path.join(os.getcwd(), '..', 'data')
files_in_data_folder = os.listdir(data_folder_path)

for file_name in files_in_data_folder:
    if file_name.endswith('associations.parquet'):
        df_name = file_name.replace('.parquet', '')
        file_path = os.path.join(data_folder_path, file_name)
        globals()[df_name] = pd.read_parquet(file_path)
        print(df_name)


odis_refugee_associations


In [50]:
files_in_data_folder

['odis_ccas.parquet',
 '.DS_Store',
 'odis_communes.parquet',
 'odis_formations_agg.parquet',
 'odis_referentiels.parquet',
 'odis_metiers_agg.parquet',
 'odis_communes_pre.parquet',
 'odis_associations_agg.parquet',
 'odis_bassins_de_vie.parquet',
 'odis_pois.parquet']

In [52]:
odis_pois.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275393 entries, 0 to 275392
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype   
---  ------    --------------   -----   
 0   id        275393 non-null  object  
 1   name      275393 non-null  object  
 2   type      275393 non-null  category
 3   category  275393 non-null  category
 4   lat       275097 non-null  float32 
 5   lon       275097 non-null  float32 
 6   codgeo    275393 non-null  category
dtypes: category(3), float32(2), object(2)
memory usage: 7.7+ MB


In [57]:
odis_pois[odis_pois.name.str.contains("wimoov", na=False, case=False)].drop_duplicates(subset=["lat", "lon"]).head(20)

,id,name,type,category,lat,lon,codgeo
43720,1fca05b5a69733029116ead3a0530abd,RSA - Plateforme mobilité WIMOOV - Aix / Gardanne,mobilite--acceder-a-un-vehicule,incl_services,43.493237,5.369391,13001
140363,bccb1bd8a137bebb0fc0fccf0c44fbab,RSA - Plateforme mobilité WIMOOV - Salon-Berre / Arles / Istres-Marignane-Vitrolles-Martigues,mobilite--acceder-a-un-vehicule,incl_services,43.652000,5.099365,13103


In [32]:
old_odis = pd.read_parquet('../data/odis_june_2025_jacques.parquet')

In [79]:
odis_referentiels[odis_referentiels.code == "11069"]

,key,code,label,reg_code
4500,communes,11069,Carcassonne,None
36076,bassins_de_vie,11069,Carcassonne,None


In [82]:
odis_refugee_associations[odis_refugee_associations.codgeo.str.contains("11069", na=False)].head()


,id,codgeo,bassin_de_vie,name,description,waldec_code
2660,W111008950,11069,11069,UKRAINE ESPOIR AUDE,"aider de tout ordre, de toutes maniere et lassistance aux refugies du conflit ukrainien par tout moyen daction autorise. Lassociation pourra etre amenee occasionnellement a organiser des evenements commerciaux, ou des activites economiques code du commerce article L442-7",020020


In [ ]:
odis_finess[(odis_finess['Departement'] == '13') & (odis_finess['Commune'] == '103')][['LibelleCategorie','nofinesset']].groupby('LibelleCategorie').count().sort_values(by='nofinesset', ascending=False)

,nofinesset
LibelleCategorie,
Pharmacie d'Officine,14
Service autonomie aide (SAA),10
Laboratoire de Biologie Médicale,6
Etablissement d'hébergement pour personnes âgées dépendantes,5
Institut Médico-Educatif (I.M.E.),4
Service de Soins Infirmiers A Domicile (S.S.I.A.D),3
Centre Hospitalier Spécialisé lutte Maladies Mentales,3
Centre Hébergement & Réinsertion Sociale (C.H.R.S.),3
"Autre Résidence Sociale (hors Maison Relais, Pension de Fami",3


# Filter Asso Réfugies

In [3]:
asso =pd.read_parquet('../pipeline/cache/raw/associations.parquet')

In [20]:
asso.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2215247 entries, 0 to 2215246
Data columns (total 39 columns):
 #   Column           Dtype 
---  ------           ----- 
 0   id               object
 1   id_ex            object
 2   siret            object
 3   rup_mi           object
 4   gestion          object
 5   date_creat       object
 6   date_decla       object
 7   date_publi       object
 8   date_disso       object
 9   nature           object
 10  groupement       object
 11  titre            object
 12  titre_court      object
 13  objet            object
 14  objet_social1    object
 15  objet_social2    object
 16  adrs_complement  object
 17  adrs_numvoie     object
 18  adrs_repetition  object
 19  adrs_typevoie    object
 20  adrs_libvoie     object
 21  adrs_distrib     object
 22  adrs_codeinsee   object
 23  adrs_codepostal  object
 24  adrs_libcommune  object
 25  adrg_declarant   object
 26  adrg_complemid   object
 27  adrg_complemgeo  object
 28  adrg_libvoie

In [61]:
asso[asso['titre_court'].str.contains('wimoov', case=False, na=False)].assign(objet=lambda x: x['objet'].str[:300])[['id', 'titre_court', 'objet_social1', 'objet', 'adrs_codeinsee']]

,id,titre_court,objet_social1,objet,adrs_codeinsee
1324302,W751186914,WIMOOV,024045,"promouvoir et initier le developpement de nouvelles pratiques de mobilite sensibiliser et accompagner tous les publics vers une mobilite autonome, responsable et respectueuse de lenvironnement",75111


In [38]:
asso[(asso.objet_social1=='019025') & (asso.adrs_codeinsee=='33063')].titre_court.head(25)

468278    RESEAU UNIVERSITAIRE AIME
469936        LA FORCE DUNE MERE FM
477263              PONT DE LESPOIR
Name: titre_court, dtype: object

In [ ]:
asso_refug = asso[asso.objet_social1.str.startswith(('003', '019', '020', '014'), na=False)].loc[
    asso['objet'].str.contains('asil', regex=False, case=False, na=False)
    |asso['objet'].str.contains('refug', regex=False, case=False, na=False)
    |asso['objet'].str.contains('nouveaux arrivants', regex=False, case=False, na=False)
    |asso['objet'].str.contains('migra', regex=False, case=False, na=False),
    ['id', 'adrs_codeinsee', 'titre','position','objet_social1', 'objet']
]


In [24]:
mapping = odis_referentiels[odis_referentiels['key'] == 'waldec_codes'].set_index('code')['label']
asso_refug['objet_lib'] = asso_refug['objet_social1'].astype(int).astype(str).map(mapping)


In [25]:
asso_refug.groupby(['objet_social1', 'objet_lib']).count().sort_values(by='id', ascending=False)

,,id,adrs_codeinsee,titre,position,objet
objet_social1,objet_lib,,,,,
014035,groupements d'entraide et de solidarité,861,861,861,861,861
020000,"ASSOCIATIONS CARITATIVES, HUMANITAIRES, AIDE AU DÉVELOPPEMENT, DÉVELOPPEMENT DU BÉNÉVOLAT",800,800,800,800,800
014000,"AMICALES, GROUPEMENTS AFFINITAIRES, GROUPEMENTS D'ENTRAIDE (HORS DÉFENSE DE DROITS FONDAMENTAUX)",772,772,772,772,772
014040,"amicale de personnes originaires d'un même pays (hors défense des droits des étrangers), d'une même région du monde",536,536,536,536,536
019025,aide aux réfugiés et aux immigrés (hors défense de droits fondamentaux),357,357,357,357,357
020020,associations caritatives intervenant au plan international,325,325,325,325,325
019000,INTERVENTIONS SOCIALES,313,313,313,313,313
003050,"défense de droits de personnes étrangères ou immigrées, de personnes réfugiées",298,298,298,298,298
020015,associations caritatives à but multiple,262,262,262,262,262


In [38]:
asso_refug.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5695 entries, 2462 to 2214945
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              5695 non-null   object
 1   adrs_codeinsee  5695 non-null   object
 2   titre           5695 non-null   object
 3   position        5695 non-null   object
 4   objet_social1   5695 non-null   object
 5   objet           5695 non-null   object
 6   objet_lib       5695 non-null   object
dtypes: object(7)
memory usage: 355.9+ KB


In [40]:
asso_refug[asso_refug.adrs_codeinsee == "33063"][['titre', 'objet_social1']].sort_values(by='objet_social1')

,titre,objet_social1
447276,MEMOIRES ET PARTAGES,003000
450015,LAMICALE DES TUNISIENS DE BORDEAUX,003015
462115,ASSOCIATION DES PARENTS DE FAMILLES ESPAGNOLES EMIGREES EN FRANCE - APFEEF DE,003015
472916,KHAIMA QUEER,003025
457113,ACCOMPAGNEMENT PSYCHOLOGIQUE ET MEDIATION INTERCULTURELLE - AMI,003040
463913,WELCOME BORDEAUX RESEAU DHOSPITALITE POUR DES DEMANDEURS DASILE DANS LA REGION DE BORDEAUX,003050
468339,FEARLESS CULTURE,014035
472166,COLLECTIF DES MIGRANT.E.S. DE BORDEAUX CMB,014035
473387,MEDIATION INTERCULTURELLE EN MILIEUX MEDICAL ET SOCIAL INTERPRETARIAT,014035
473665,AQUITAINE EDUCATION - AQUEDUC,014035


In [32]:
odis_referentiels[(odis_referentiels.code=='23000')].head(100)

,key,code,label,reg_code
850,waldec_codes,23000,"REPRÉSENTATION, PROMOTION ET DÉFENSE D'INTÉRÊTS ÉCONOMIQUES",None


In [33]:
odis_associations_agg.head()

,codgeo,id_waldec,count
0,00000,001000,1
1,00000,002000,1
2,00000,005000,2
3,00000,006000,1
4,00000,006030,3


## Filter RNA Mini ODIS

In [93]:
asso.head()

,id,id_ex,siret,rup_mi,gestion,date_creat,date_decla,date_publi,date_disso,nature,groupement,titre,titre_court,objet,objet_social1,objet_social2,adrs_complement,adrs_numvoie,adrs_repetition,adrs_typevoie,adrs_libvoie,adrs_distrib,adrs_codeinsee,adrs_codepostal,adrs_libcommune,adrg_declarant,adrg_complemid,adrg_complemgeo,adrg_libvoie,adrg_distrib,adrg_codepostal,adrg_achemine,adrg_pays,dir_civilite,siteweb,publiweb,observation,position,maj_time
0,W431000001,None,None,None,431S,2006-03-10,2006-03-10,0001-01-01,0001-01-01,D,S,MENTION TRES BIEN,MENTION TRES BIEN,aide et soutien scolaire,015025,000000,None,12,,RUE,de La Rodde,None,43258,43360,Vergongheon,None,None,None,12 Rue de La Rodde,None,43360,VERGONGHEON,FRANCE,PM,None,0,None,A,2007-10-09 15:23:51
1,W431000002,None,None,None,431S,2006-03-03,2013-02-27,0001-01-01,0001-01-01,D,S,GROUPEMENT DEMPLOYEURS DU LIVRADOIS,GROUPEMENT DEMPLOYEURS DU LIVRADOIS,Mettre un salarie a disposition de ses adherents - Embauche de salaries a plusieurs - Reduire des couts de main doeuvre,030005,000000,None,None,,None,LE BOURG,None,43128,43160,Malvieres,None,None,None,LE BOURG,None,43160,MALVIERES,FRANCE,PM,None,0,None,A,2013-02-27 09:14:40
2,W431000003,None,None,None,431S,2006-02-27,2006-02-27,0001-01-01,0001-01-01,D,S,GROUPEMENT DEMPLOYEURS DU CANTOU,GROUPEMENT DEMPLOYEURS DU CANTOU,La mise a disposition de ses membres dun salarie lie a ce groupement par un contrat de travail,030000,000000,None,None,,None,CHIRAC,None,43056,43300,Chanteuges,None,None,None,CHIRAC,None,43300,CHANTEUGES,FRANCE,PM,None,0,None,A,2008-08-11 20:09:10
3,W431000004,None,None,None,431S,2006-02-22,2020-02-05,0001-01-01,0001-01-01,D,S,ASSOCIATION DES PARENTS DELEVES DES ECOLES PUBLIQUES JULES FERRY DE LANGEAC,ASSOCIATION DES PARENTS DELEVES D...,"Toute activite susceptible dapporter un soutien utile a la vie de lecole, une collaboration efficace a laction des maitres et veiller a la defense des interets materiels et moraux de lecole",015065,000000,Mairie,None,,None,None,None,43112,43300,Langeac,None,Mairie,None,None,None,43300,LANGEAC,FRANCE,PF,None,0,None,A,2020-02-11 11:12:25
4,W431000005,None,None,None,431S,2006-02-16,2006-02-16,0001-01-01,0001-01-01,D,S,LABRIPOM,LABRIPOM,"Cette association a pour but la creation, la production, la representation et lorganisation de spectacle et de toutes manifestations culturelles et de loisirs.",006030,000000,None,18,,RUE,Saint Verny,None,43040,43100,Brioude,None,None,None,18 Rue Saint Verny,None,43100,BRIOUDE,FRANCE,PF,None,0,None,A,2008-03-04 17:31:39


In [ ]:
waldec_prefixes = ["003", "018", "019", "020", "032"] #014040
cols_to_keep = ['id', 'adrs_codeinsee', 'objet_social1','titre_court', 'objet']
asso_mini = asso[(asso['objet_social1'].str[:3].isin(waldec_prefixes)) & (asso.position == 'A')][cols_to_keep]
asso_mini['objet'] = asso_mini['objet'].str[:250].str.lower()
asso_mini.to_parquet('../data/odis_asso_mini.parquet', compression='brotli', index=False)


In [117]:
asso_mini.head(50)

,id,adrs_codeinsee,objet_social1,objet
55,W431000057,43132,020015,"organisation des manifestations caritatives comme le telethon, la marche adot43, adapei"
65,W431000067,43264,019040,"louverture et la gestion dun lieu de vie, sis a villeneuve dallier, accueillant des adolescents et pre-adolescents filles et garcons en difficultes, confies par decision de justice ou par les organismes departementaux."
178,W431000184,43040,020020,"encourager et cultiver lideal de servir, le developpement des relations personnelles damour entre ses membres en vue de leur fournir des occasions de servir linteret general, lobservation des regles de probite et de delicatesse dans lexercice de leur"
208,W431000217,43040,019020,favoriser linsertion sociales et professionnelle des personnes en difficulte par lexercice dune activite professionnelle
231,W431000240,43118,020015,projets humanitaires.
263,W431000277,43067,003040,denoncer les injustices et les mensonges qui causent des victimes innocentes et sans defense - aider ces victimes.
295,W431000313,43112,018050,favoriser linsertion des personnes handicapees par la promotion dactivites a caractere culturel en collaboration avec dautres associations ou en utilisant ses propres moyens
364,W431000388,43040,020015,"laide materielle et morale aux oeuvres privees, charitables, hospitalieres, sanitaires et sociales, et en particulier ladministration et la gestion des residences saint dominique a brioude et la maison de retraite saint-dominique a craponne-sur-arzon"
410,W431000434,43258,018050,"lorientation, la reeducation et la formation professionnelle des personnes en situation de handicap, leur insertion ou reinsertion dans la vie economique et sociale afin de concourir a leur integration dans la societe informer lopinion sur ces sujet"
536,W431000561,43040,018005,"de faire decouvrir et promouvoir le bien-etre des enfants massage bebe, portage, langue des signes... et des adultes pilates..."
